In [1]:
# LLM Based Routing

In [2]:
from IPython.display import Image
from IPython.core.display import HTML
import html
import copy
import re
import os
import json
from datetime import datetime
import PyPDF2
from langchain.text_splitter import RecursiveCharacterTextSplitter
from cfggenaisdk import CFGGenAIGDK

In [3]:
gdk = CFGGenAIGDK("uc0053_qopt_gen_qa","TEST_GENAI_SDK","lets strat tesing exp gdk", verbose=True)
LLM_MODEL_ID = "md0005_openai_gpt4o"
FILE = "PhysicalDataModelStandards.pdf"  # Standards document

***********YOUR EXP ID - exp13715_uc0053_qopt_gen_qa*************


In [4]:
# Rajneesh Code

# =========================================================
# 2. HELPER UTILITIES (Defined at top-level to avoid Scope Errors)
# =========================================================

def initialization_rag():
    """Reads and chunks the Standards PDF."""
    print("🔍 [PHASE: RAG] Reading Standards PDF...")
    try:
        if not os.path.exists(FILE):
            print(f"⚠️ {FILE} not found. Using general SQL knowledge.")
            return []
        full_text = ""
        with open(FILE, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text = page.extract_text()
                if text: full_text += text + "\n"
        
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
        return splitter.split_text(full_text)
    except Exception as e:
        print(f"❌ [Error] PDF Load Failed: {e}")
        return []

def run_policy_gate(sql):
    """Validates SQL safety and determines the processing path."""
    print("🛡️ [PHASE: GATE] Validating SQL Safety...")
    sql_clean = sql.strip().upper()
    # Define destructive commands
    destructive = r"\b(DROP|TRUNCATE|DELETE|ALTER|GRANT|REVOKE|VACUUM)\b"
    if re.search(destructive, sql_clean):
        return "BLOCK"
    
    # Check if it's a read-only query
    if sql_clean.startswith("SELECT") or sql_clean.startswith("WITH"):
        return "OPTIMIZE_PATH"
    return "SUGGESTION_ONLY_PATH"

def semantic_search(user_sql, chunks):
    """Finds relevant chunks from the PDF context."""
    print("🧠 [PHASE: SEMANTIC] Searching PDF context...")
    if not chunks:
        return "Corporate SQL Standards: Enforce naming conventions and audit columns."
    relevant_chunks = [c for c in chunks if any(word.lower() in c.lower() for word in user_sql.split()[:8])]
    return "\n---\n".join(relevant_chunks[:3])

def extract_final_output(response_text, path):
    """Cleans the LLM output based on the path taken."""
    if path == "SUGGESTION_ONLY_PATH":
        return "POLICY NOTICE: Direct SQL optimization is disabled for DDL/DML. Review analysis for suggestions."
    
    matches = re.findall(r"```sql\s+(.*?)\s+```", response_text, re.DOTALL | re.IGNORECASE)
    return matches[-1].strip() if matches else "No optimized SQL block generated."

# =========================================================
# 3. USE CASE HANDLER (The Dispatcher Target)
# =========================================================

def handle_query_optimizer(summary_payload: dict):
    print("✅ [PHASE: AGENT ACTIVATED] Routed to: SQL Query Optimizer")
    
    # Extract the SQL from the summarized payload
    user_sql = summary_payload.get("raw_context", {}).get("original_question", "").strip()
    
    if not user_sql:
        print("❌ [Error] No SQL found in the summarized payload.")
        return

    # STEP 1: Policy Gate
    path = run_policy_gate(user_sql)
    if path == "BLOCK":
        print("🛑 [SECURITY] Destructive command blocked. Operation halted.")
        return

    # STEP 2: RAG Context
    chunks = initialization_rag()
    context = semantic_search(user_sql, chunks)

    # STEP 3: Prompt Construction
    if path == "OPTIMIZE_PATH":
        task_instruction = (
            "Task: Optimize the SELECT query for performance and sargability. "
            "Identify naming violations and missing audit columns (e.g., CREATE_BY_NM). "
            "Provide the final optimized SQL inside ```sql blocks."
        )
    else:
        task_instruction = (
            "Task: DDL/Creation query detected. DO NOT provide an optimized code block. "
            "Instead, list compliance violations and provide text suggestions only."
        )

    system_prompt = f"Role: Senior SQL Expert.\nStandards Context: {context}\n{task_instruction}"
    
    print(f"🤖 [PHASE: LLM] Generating solution for {path}...")
    prompt = {
        "prompt_template": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Query: {user_sql}"}
        ]
    }
    
    # LLM Call
    raw_response = gdk.invoke_llmgateway(prompt, {"temperature": 0}, LLM_MODEL_ID)
    
    try:
        final_solution = raw_response['genResponse']['choices'][0]['message']['content']
    except Exception:
        final_solution = str(raw_response)

    # Display Analysis
    print("\n" + "-"*30 + " ANALYSIS " + "-"*30)
    print(final_solution)
    
    # Display Final Result
    print("\n" + "="*50)
    print("   FINAL ACTIONABLE OUTPUT")
    print("="*50)
    print(extract_final_output(final_solution, path))
    print("="*50)


In [5]:
# Satheesh Code

# =========================================================
# Config
# =========================================================
MODEL_ID = "md0005_openai_gpt4o"

ROUTER_HYPERPARAM = {
    "max_tokens": 250,
    "temperature": 0.2
}

SUMMARIZER_HYPERPARAM = {
    "max_tokens": 200,
    "temperature": 0.2
}

MAX_FOLLOW_UPS = 5  # safety guard to avoid infinite loops

USE_CASES = [
    "Database Health Monitoring",
    "DML/DDL Query Optimizer",
    "Kafka Assist",
    "Provisioning Development Databases",
    "Database Recommendation",
]

SAMPLE_QUESTIONS = [
    "Why is my database CPU spiking every morning around 9 AM?",
    "Can you optimize this SQL query? EXPLAIN shows a sequential scan.",
    "My Kafka consumer lag is growing—how do I troubleshoot rebalance issues?",
    "Can you provision a new dev database cloned from staging with masked data?",
    "Which database should I choose for time-series + high write throughput?",
]

WELCOME_MESSAGE = f"""
👋 Welcome! I can route your request to the right database platform assistant.

✅ Supported use cases today:
1) Database Health Monitoring
2) DML/DDL Query Optimizer
3) Kafka Assist
4) Provisioning Development Databases
5) Database Recommendation

Ask your question in plain English (or paste SQL / Kafka details).
"""

# =========================================================
# Router Prompt Template
# =========================================================
ROUTER_PROMPT_TEMPLATE = {
    "prompt_template": [
        {
            "role": "system",
            "content": """
You are a Database Platform Router Assistant that routes user questions to the correct supported use case.

**IMPORTANT (Conversation Start Behavior)**: Begin every conversation by either:
1. If the user message clearly contains a database/Kafka/platform request with an action, symptom, or goal: Start routing immediately.
2. If the user message is unclear or just a label (e.g., "Kafka", "SQL", "database", "recommendation"): Ask a clarifying follow-up question using action=ask_follow_up.

**RESPONSE STYLE**: Keep responses concise, professional, and focused. No unnecessary explanations or chattiness.

<ROLE>
Act as an intelligent routing agent for a database platform assistant. Analyze the user's input and determine the correct supported use case. If the user input is too vague, ask exactly one follow-up question to clarify intent before routing.
</ROLE>

<TASK>
Route the user's request to one of the supported use cases:
1) Database Health Monitoring
2) DML/DDL Query Optimizer
3) Kafka Assist
4) Provisioning Development Databases
5) Database Recommendation

If the request is ambiguous or missing key details, ask exactly one targeted follow-up question.
If the request is out-of-domain, return invalid and provide supported use cases + sample questions.
</TASK>

<CONTEXT>
You help users who want to:
- Diagnose database performance issues and operational health symptoms
- Optimize SQL DML/DDL queries and improve execution plans
- Troubleshoot Kafka producers/consumers, lag, topics, partitions, offsets, and cluster behaviors
- Provision/clone/refresh non-production development databases and environments
- Choose the best database technology for a workload (recommendation within supported scope)
</CONTEXT>

<SUPPORTED_USE_CASES>
1. **Database Health Monitoring**: Performance degradation, CPU/memory/IO spikes, latency, SLA/SLO, monitoring, alerts, anomalies, trends
2. **DML/DDL Query Optimizer**: SQL query tuning, DML/DDL improvements, indexing/partitioning suggestions, EXPLAIN plan analysis
3. **Kafka Assist**: Consumer lag, rebalances, producer/consumer config, topics, partitions, offsets, brokers, errors
4. **Provisioning Development Databases**: Create/clone/refresh dev/test/stage DBs, sandbox environments, data masking requirements
5. **Database Recommendation**: Choosing a database based on workload, constraints, platform comparisons
</SUPPORTED_USE_CASES>

<ROUTING_GUIDELINES>
Use these signals to determine the best route:
- If the user provides SQL or asks to optimize/tune/EXPLAIN a query or DML/DDL -> **DML/DDL Query Optimizer**
- If the user mentions Kafka topics/consumer lag/producer/brokers/offsets/rebalance OR Kafka configuration/troubleshooting -> **Kafka Assist**
- If the user reports DB performance symptoms (latency/spikes/CPU/memory/IO/locks/monitoring/alerts/SLA) -> **Database Health Monitoring**
- If the user asks to create/clone/refresh dev/non-prod DBs or needs env provisioning -> **Provisioning Development Databases**
- If the user asks which database to choose for a workload, compares engines/platforms -> **Database Recommendation**
- If unrelated to database/Kafka/provisioning/recommendation -> **Invalid**
</ROUTING_GUIDELINES>

<MINIMAL_INPUT_RULES>
If the user's input is too vague, do NOT immediately route. Instead, ask one follow-up question.

Treat the input as "too vague" when:
- It is a single word or short label, e.g., "Kafka", "SQL", "database", "monitoring", "recommendation"
- It only names a technology/use case without a goal, action, or symptom
- It lacks the minimum detail needed to pick a path confidently

Examples (must ask follow-up):
- "Kafka"
- "SQL"
- "Need DB help"
- "Monitoring"
- "Provisioning"

Follow-up behavior for vague inputs:
- action must be "ask_follow_up"
- use_case must be your best-guess category (e.g., "Kafka Assist" for "Kafka")
- Ask exactly ONE clarifying question that identifies the user's goal/action.

Kafka-specific vague input example:
- If user says "Kafka", ask: "What would you like to do with Kafka—troubleshoot consumer lag, create/manage topics, tune producer/consumer configs, or investigate broker issues?"
</MINIMAL_INPUT_RULES>

<FOLLOW_UP_RULES>
Ask a follow-up question ONLY when:
- The request could belong to multiple supported use cases, OR
- The question is missing key details needed to route, OR
- The input matches <MINIMAL_INPUT_RULES>

Follow-up requirements:
- Ask exactly ONE question
- The question must be targeted and disambiguating
- Do NOT request credentials, secrets, or sensitive data
</FOLLOW_UP_RULES>

<OUT_OF_SCOPE_HANDLING>
If out-of-domain, respond as Invalid and include:
- The supported use cases list
- At least 3 sample questions (from <SAMPLE_QUESTIONS>)
</OUT_OF_SCOPE_HANDLING>

<SAMPLE_QUESTIONS>
- "Why is my database CPU spiking every morning around 9 AM?"
- "Can you optimize this SQL query? EXPLAIN shows a sequential scan."
- "My Kafka consumer lag is growing—how do I troubleshoot rebalance issues?"
- "Can you provision a new dev database cloned from staging with masked data?"
- "Which database should I choose for time-series + high write throughput?"
</SAMPLE_QUESTIONS>

<OUTPUT_FORMATS>
Return VALID JSON ONLY (no markdown, no extra text) in exactly this schema:

{
  "action": "route" | "ask_follow_up" | "invalid",
  "use_case": "Database Health Monitoring" | "DML/DDL Query Optimizer" | "Kafka Assist" | "Provisioning Development Databases" | "Database Recommendation" | "Invalid",
  "confidence": 0.0-1.0,
  "reason": "short reason",
  "follow_up_question": "string (required only if action=ask_follow_up)"
}

Constraints:
- If action="route": confidence >= 0.75 and follow_up_question must be omitted or empty
- If action="ask_follow_up": confidence < 0.75 and follow_up_question must be present (exactly one question)
- If action="invalid": use_case must be "Invalid" and confidence >= 0.75
</OUTPUT_FORMATS>

<GUARDRAILS_AND_OPERATIONAL_CONSTRAINTS>
Scope Lock:
- Only use the supported use cases listed in <SUPPORTED_USE_CASES>. Do not invent new categories.

Determinism:
- Choose exactly one use case when routing.
- Ask only one follow-up question at a time when needed.

No Sensitive Requests:
- Do not ask for passwords, tokens, private keys, credentials, or confidential access details.

No Extra Text:
- Output must be JSON only, matching the schema in <OUTPUT_FORMATS>.

Ambiguity Handling:
- Prefer ask_follow_up when multiple routes are plausible.
- For vague single-word inputs, ALWAYS ask follow-up (per <MINIMAL_INPUT_RULES>).

Time/Locale:
- If dates/times are referenced, use Asia/Kolkata timezone.
</GUARDRAILS_AND_OPERATIONAL_CONSTRAINTS>

Start Behavior Reminder:
- On conversation start, apply the "IMPORTANT (Conversation Start Behavior)" rules verbatim.
"""
        },
        {
            "role": "user",
            "content": """
{question}
"""
        }
    ]
}

def build_router_prompt(question: str) -> dict:
    p = copy.deepcopy(ROUTER_PROMPT_TEMPLATE)
    p["prompt_template"][1]["content"] = p["prompt_template"][1]["content"].format(question=question)
    return p

# =========================================================
# Summarizer Prompt Template
#   - Summarizes original question + follow-up Q/A
#   - Produces a payload to send to the downstream agent
# =========================================================
SUMMARIZER_PROMPT_TEMPLATE = {
    "prompt_template": [
        {
            "role": "system",
            "content": """
You are a summarization assistant preparing a clean, structured handoff to a specialized platform agent.

Given:
- chosen use case
- original user question
- (optional) follow-up question asked by the router
- (optional) user's follow-up answer

Create a concise summary that the downstream agent can use immediately.

Output MUST be valid JSON only (no markdown, no extra text) in this schema:

{
  "use_case": "<the provided use case>",
  "user_intent_summary": "1-2 sentences in the question format",
  "key_details": ["bullet", "bullet", "..."],
  "assumptions": ["only if needed"],
  "missing_info": ["only if still required for the agent"],
  "raw_context": {
    "original_question": "...",
    "follow_up_question": "...",
    "follow_up_answer": "..."
  }
}

Rules:
- Do not invent details.
- If SQL is included, keep it in raw_context but do not paste full SQL into user_intent_summary.
- Keep key_details short and concrete.
"""
        },
        {
            "role": "user",
            "content": """
Use case:
{use_case}

Original question:
{original_question}

Follow-up question (if any):
{follow_up_question}

Follow-up answer (if any):
{follow_up_answer}
"""
        },
    ]
}

def build_summarizer_prompt(use_case: str, original_question: str, follow_up_question: str, follow_up_answer: str) -> dict:
    p = copy.deepcopy(SUMMARIZER_PROMPT_TEMPLATE)
    p["prompt_template"][1]["content"] = p["prompt_template"][1]["content"].format(
        use_case=use_case,
        original_question=original_question or "",
        follow_up_question=follow_up_question or "",
        follow_up_answer=follow_up_answer or "",
    )
    return p

# =========================================================
# Utilities (JSON extractor + helpers)
# =========================================================
def get_llm_content(resp: dict) -> str:
    content = resp["genResponse"]["choices"][0]["message"]["content"]
    return html.unescape(content).strip()

def safe_json_load(text: str):
    """
    Robust JSON extraction if the model accidentally includes extra text.
    """
    text = html.unescape(text).strip()

    # 1) direct JSON parse
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2) extract first JSON object using regex
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

    return None

def show_invalid_message():
    print("\n❌ Sorry—this question looks out of scope for the assistants supported today.\n")
    print("✅ Supported use cases:")
    for i, uc in enumerate(USE_CASES, start=1):
        print(f"  {i}) {uc}")

    print("\n💡 Sample questions you can ask:")
    for s in SAMPLE_QUESTIONS:
        print(f"  - {s}")
    print()

# =========================================================
# Placeholder Agents (NO implementation)
#   - These receive the summarized payload
# =========================================================
def handle_database_health_monitoring(summary_payload: dict):
    print("✅ Routed to: Database Health Monitoring (placeholder)")
    print("Summary payload:\n", json.dumps(summary_payload, indent=2), "\n")



def handle_kafka_assist(summary_payload: dict):
    print("✅ Routed to: Kafka Assist (placeholder)")
    print("Summary payload:\n", json.dumps(summary_payload, indent=2), "\n")

def handle_provisioning_dev_db(summary_payload: dict):
    print("✅ Routed to: Provisioning Development Databases (placeholder)")
    print("Summary payload:\n", json.dumps(summary_payload, indent=2), "\n")

def handle_database_recommendation(summary_payload: dict):
    print("✅ Routed to: Database Recommendation (placeholder)")
    print("Summary payload:\n", json.dumps(summary_payload, indent=2), "\n")

def dispatch(use_case: str, summary_payload: dict):
    if use_case == "Database Health Monitoring":
        return handle_database_health_monitoring(summary_payload)
    if use_case == "DML/DDL Query Optimizer":
        return handle_query_optimizer(summary_payload)
    if use_case == "Kafka Assist":
        return handle_kafka_assist(summary_payload)
    if use_case == "Provisioning Development Databases":
        return handle_provisioning_dev_db(summary_payload)
    if use_case == "Database Recommendation":
        return handle_database_recommendation(summary_payload)

    # Anything else considered invalid
    return show_invalid_message()

# =========================================================
# Router + Follow-up + Summarize + Dispatch (Core)
# =========================================================
def route_with_followup_and_dispatch(initial_question: str):
    original_question = initial_question
    follow_up_question = ""
    follow_up_answer = ""
    followups_used = 0

    # Step 1: route (and possibly ask follow-up)
    current_input = original_question

    while True:
        prompt = build_router_prompt(current_input)
        resp = gdk.invoke_llmgateway(prompt, ROUTER_HYPERPARAM, MODEL_ID)

        router_text = get_llm_content(resp)
        decision = safe_json_load(router_text)

        if not decision:
            # Malformed output => safe fallback
            show_invalid_message()
            return

        action = (decision.get("action") or "").strip()
        use_case = (decision.get("use_case") or "Invalid").strip()
        confidence = float(decision.get("confidence", 0.0))
        reason = (decision.get("reason") or "").strip()
        fq = (decision.get("follow_up_question") or "").strip()

        # Invalid/out-of-domain
        if action == "invalid" or use_case == "Invalid":
            show_invalid_message()
            return

        # Follow-up requested
        if action == "ask_follow_up":
            followups_used += 1
            follow_up_question = fq or "Can you provide a bit more detail so I can route this correctly?"
            print(f"\n🤔 {follow_up_question}\n")
            follow_up_answer = input("Your answer: ").strip()

            if followups_used >= MAX_FOLLOW_UPS:
                # Stop infinite loops: proceed with best guess use_case even if low confidence
                break

            # Re-route with combined context to improve accuracy
            current_input = (
                f"Original question: {original_question}\n"
                f"Follow-up question: {follow_up_question}\n"
                f"Follow-up answer: {follow_up_answer}"
            )
            continue

        # Route
        if action == "route":
            break

        # Unknown action => fallback
        show_invalid_message()
        return

    # Step 2: Summarize the original + follow-up context for downstream agent
    summarize_prompt = build_summarizer_prompt(
        use_case=use_case,
        original_question=original_question,
        follow_up_question=follow_up_question,
        follow_up_answer=follow_up_answer
    )
    sum_resp = gdk.invoke_llmgateway(summarize_prompt, SUMMARIZER_HYPERPARAM, MODEL_ID)

    sum_text = get_llm_content(sum_resp)
    summary_payload = safe_json_load(sum_text)

    if not summary_payload:
        # If summarizer returns malformed JSON, create a minimal safe payload
        summary_payload = {
            "use_case": use_case,
            "user_intent_summary": "Unable to parse summarizer output. Passing raw context.",
            "key_details": [],
            "assumptions": [],
            "missing_info": [],
            "raw_context": {
                "original_question": original_question,
                "follow_up_question": follow_up_question,
                "follow_up_answer": follow_up_answer
            }
        }

    # Step 3: Dispatch to placeholder agent (passing the summarized payload)
    print(f"\n➡️ Final Route: {use_case}\n")
    dispatch(use_case, summary_payload)

# =========================================================
# CLI Driver
# =========================================================
def main():
    print(WELCOME_MESSAGE)
    while True:
        q = input("\nEnter your question (or type 'exit'): ").strip()
        if q.lower() in ("exit", "quit"):
            print("\n👋 Bye! Have a great day.\n")
            break
        if not q:
            continue

        route_with_followup_and_dispatch(q)

if __name__ == "__main__":
    main()


👋 Welcome! I can route your request to the right database platform assistant.

✅ Supported use cases today:
1) Database Health Monitoring
2) DML/DDL Query Optimizer
3) Kafka Assist
4) Provisioning Development Databases
5) Database Recommendation

Ask your question in plain English (or paste SQL / Kafka details).




Enter your question (or type 'exit'):  exit



👋 Bye! Have a great day.

